# Phase 3-1 — Groq API 래퍼 (_groq.py)

**목표:** Python 표준 라이브러리 `urllib`로 외부 HTTPS API를 직접 호출한다.

**이 노트북을 마치면:**
- [ ] urllib.request.Request로 POST 요청을 구성할 수 있다
- [ ] Authorization Bearer 헤더를 올바르게 추가할 수 있다
- [ ] 응답 헤더에서 rate limit 정보를 파싱할 수 있다
- [ ] 429 응답에서 대기 시간을 파싱해 재시도할 수 있다

**완성 후 연결:**  
노트북에서 구현한 패턴을 `src/_groq.py`에 옮기기

## 섹션 1 — urllib로 HTTP 요청 구성

### urllib.request.Request

Python 표준 라이브러리로 HTTP 요청 객체를 만드는 방법.

```python
import urllib.request

req = urllib.request.Request(
    url,
    data=b"...",             # bytes (POST body)
    headers={"Key": "Val"},  # HTTP 헤더
    method="POST",
)
```

### JSON payload 만들기

```python
import json
payload = {"model": "llama-3.1-8b-instant", "messages": [...]}
data = json.dumps(payload).encode("utf-8")   # str → bytes
```

### 응답 읽기

```python
with urllib.request.urlopen(req, timeout=60) as resp:
    body_str = resp.read().decode("utf-8")   # bytes → str
    body = json.loads(body_str)              # str → dict
```

### 위밍업


In [ ]:
import json
import urllib.request
import os

# .env에서 키 읽기 (직접 넣어도 됨 — 노트북에서만, 소스코드에는 금지)
def _read_env_key():
    from pathlib import Path
    env = Path("../.env")
    if env.exists():
        for line in env.read_text(encoding="utf-8").splitlines():
            if line.startswith("GROQ_API_KEY=") and not line.startswith("#"):
                return line[len("GROQ_API_KEY="):].strip().strip('"').strip("'")
    return os.environ.get("GROQ_API_KEY", "")

API_KEY = _read_env_key()
print("키 로드:", "OK" if API_KEY else "MISSING")

In [ ]:
# 완성된 예제 — 직접 실행해보고 패턴을 익힌다
_CHAT_URL = "https://api.groq.com/openai/v1/chat/completions"
_UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

payload = json.dumps({
    "model": "llama-3.1-8b-instant",
    "messages": [{"role": "user", "content": "한 줄로 자기소개해줘"}],
    "max_tokens": 50,
}).encode("utf-8")

req = urllib.request.Request(
    _CHAT_URL,
    data=payload,
    headers={
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "User-Agent": _UA,
    },
    method="POST",
)

with urllib.request.urlopen(req, timeout=60) as resp:
    body = json.loads(resp.read().decode("utf-8"))

print(body["choices"][0]["message"]["content"])

### 미니 실습: build_headers() 구현

API 호출에 필요한 헤더 dict를 반환하는 함수를 만든다.

In [ ]:
def build_headers(api_key: str) -> dict:
    """Authorization + Content-Type + User-Agent 헤더 dict를 반환한다."""
    # TODO: 위 워밍업 예제의 headers dict를 그대로 반환
    header={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "User-Agent": _UA,
    }
    return header

In [ ]:
# 채점
h = build_headers("test_key_123")
assert isinstance(h, dict), "dict를 반환해야 합니다"
assert "Authorization" in h, "Authorization 헤더 누락"
assert h["Authorization"] == "Bearer test_key_123", "Bearer 토큰 형식 확인"
assert "Content-Type" in h, "Content-Type 헤더 누락"
assert "User-Agent" in h, "User-Agent 헤더 누락"
print("✓ 통과")

### 본 실습: make_request() 구현

URL, payload dict, api_key를 받아 urllib.request.Request 객체를 반환한다.


In [ ]:
def make_request(url: str, payload: dict, api_key: str) -> urllib.request.Request:
    """POST 요청 객체를 만들어 반환한다.
    
    힌트:
    - payload를 json.dumps().encode("utf-8") 로 직렬화
    - build_headers(api_key) 활용
    - method="POST"
    """
    # TODO
    payload = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        url,
        data=payload,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "User-Agent": _UA,
        },
        method="POST"
    )
    return req

In [ ]:
req = make_request("https://example.com/api", {"key": "val"}, "my_key")
assert isinstance(req, urllib.request.Request), "Request 객체여야 합니다"
assert req.get_method() == "POST", "method는 POST여야 합니다"
assert req.full_url == "https://example.com/api", "URL 확인"
assert req.get_header("Authorization") == "Bearer my_key", "Authorization 확인"
assert req.data == json.dumps({"key": "val"}).encode("utf-8"), "payload 직렬화 확인"
print("✓ 통과")

**연결:** `make_request()`는 `_groq.py`의 `chat_completion()` 안에서
`urllib.request.Request(...)` 을 직접 만드는 부분에 해당한다.


### 정리

```
make_request()로 Request 객체 생성
    ↓
urlopen(req, timeout=60)으로 실제 전송
    ↓
resp.read().decode("utf-8") → JSON 파싱
    ↓
content / usage / rate_limit 반환
```

## 섹션 2 — Rate Limit 헤더 파싱

응답 헤더에서 rate limit 정보를 추출해 dict로 반환한다.
이 값은 `server.py`가 응답 body에 포함시켜 `chat.js`의 배지에 표시된다.

### 워밍업

In [ ]:
# urllib의 HTTPResponse 헤더는 dict처럼 접근 가능
# 없는 키는 None 대신 기본값 반환
class MockHeaders:
    def __init__(self, d): self._d = d
    def get(self, key, default=""): return self._d.get(key, default)

mock = MockHeaders({
    "x-ratelimit-remaining-requests": "28",
    "x-ratelimit-remaining-tokens":   "5000",
    "x-ratelimit-reset-requests":     "2s",
    "x-ratelimit-reset-tokens":       "500ms",
})
print(mock.get("x-ratelimit-remaining-requests"))  # "28"
print(mock.get("x-ratelimit-limit-requests", ""))  # "" (없는 키)

### 미니 실습

In [ ]:
def parse_rate_limit(headers) -> dict:
    """응답 헤더에서 rate limit 정보를 추출해 dict로 반환한다.
    
    반환 키:
        remaining_requests, remaining_tokens,
        reset_requests, reset_tokens
    힌트: headers.get("x-ratelimit-remaining-requests", "")
    """
    # TODO
    return {
        "remaining_requests": headers.get("x-ratelimit-remaining-requests", ""),
        "remaining_tokens": headers.get("x-ratelimit-remaining-tokens"),
        "reset_requests": headers.get("x-ratelimit-reset-requests"),
        "reset_tokens": headers.get("x-ratelimit-reset-tokens")
        }

In [ ]:
rl = parse_rate_limit(mock)
assert isinstance(rl, dict), "dict를 반환해야 합니다"
assert set(rl.keys()) == {"remaining_requests", "remaining_tokens",
                           "reset_requests", "reset_tokens"}, "키 이름 확인"
assert rl["remaining_requests"] == "28", "remaining_requests 값 확인"
assert rl["reset_tokens"] == "500ms", "reset_tokens 값 확인"

# 없는 헤더는 빈 문자열
empty = MockHeaders({})
rl2 = parse_rate_limit(empty)
assert rl2["remaining_requests"] == "", "없는 헤더는 빈 문자열"
print("✓ 통과")

**연결:** `parse_rate_limit()`은 `_groq.py`의 `chat_completion()` 내부,
`urlopen()` 컨텍스트 매니저 안에서 `resp.headers`를 인자로 호출된다.


### 정리



## 섹션 3 — 429 재시도 패턴

한도 초과 시 Groq는 오류 메시지에 대기 시간을 알려준다.
```json
{"error": {"message": "Rate limit exceeded: please try again in 12.5s"}}
```
정규식으로 숫자를 추출해 `time.sleep()` 후 재시도한다.

### 워밍업

In [ ]:
import re

# 정규식으로 숫자 추출
messages = [
    "Rate limit exceeded: please try again in 12.5s",
    "Please try again in 60s",
    "Unknown error occurred",   # 숫자 없음
]

for msg in messages:
    m = re.search(r"try again in ([\d.]+)s", msg)
    if m:
        print(f"대기 시간: {float(m.group(1))}초")
    else:
        print("대기 시간 파싱 실패 → 기본값 사용")

### 본 실습

In [ ]:
def extract_wait_seconds(error_msg: str, default: float = 60.0) -> float:
    """오류 메시지에서 대기 시간(초)을 추출한다.
    
    파싱 실패 시 default 반환.
    힌트: re.search(r"try again in ([\d.]+)s", error_msg)
    """
    # TODO
    t = re.search(r"try again in ([\d.]+)s", error_msg)
    if t:
        return float(t.group(1))
    else:
        return default

In [ ]:
assert abs(extract_wait_seconds("please try again in 12.5s") - 12.5) < 1e-6, "12.5초"
assert abs(extract_wait_seconds("try again in 60s") - 60.0) < 1e-6, "60초"
assert abs(extract_wait_seconds("unknown error") - 60.0) < 1e-6, "파싱 실패 → 기본값"
assert abs(extract_wait_seconds("unknown error", default=30.0) - 30.0) < 1e-6, "기본값 변경"
print("✓ 통과")

**연결:** `extract_wait_seconds()`는 `_groq.py chat_completion()`의 429 처리 블록에서 쓰인다.

```python
# _groq.py 내부 (골격 파일 구현 시 참고)
if exc.code == 429 and attempt < 3:
    wait = extract_wait_seconds(error_msg) + 2.0   # 여유 시간 추가
    time.sleep(wait)
    continue
```
### 정리


## 스스로 정리해보기

노트북 완료 후 직접 작성:
- urllib로 외부 API를 호출할 때 꼭 필요한 헤더 3가지는?  
    - url, data, headers (X, urllib.request.Request의 파라미터임)
    - 정답 :
        Authorization — API 키 인증 (Bearer ...)
        Content-Type — 요청 body 형식 (application/json)
        User-Agent — 클라이언트 식별 (Cloudflare 우회용)

- Cloudflare가 Python을 차단하는 이유와 우회 방법은? 
    - 봇으로 인식해서 막기 때문에, _UA를 붙여서 우회한다. (-, 봇으로 인식하는 것은 맞지만 왜 봇으로 인식하는지가 빠짐.)
    - 정답 : Python은 기본적으로 Python-urllib/3.x 같은 UA를 보내는데, Cloudflare가 이를 봇 시그니처로 판단해 차단합니다. 브라우저 UA 문자열로 위장하면 통과됩니다.


- 429 오류가 발생했을 때 바로 raise하지 않고 재시도하는 이유는?
    - 대기 시간 이후 다시 요청하려고 (X, 왜 재시도가 가능한지를 적어야 함.)
    - 정답 : 429는 일시적인 한도 초과이지 영구적인 오류가 아닙니다. 서버가 "지금은 안 되지만 잠깐 기다리면 된다"고 알려주는 것이므로, 즉시 raise하면 호출 자체가 실패하고, 대기 후 재시도하면 정상 처리됩니다.